In [1]:
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
import pandas as pd
import json

from ontologx.backend import LLMFactory
from ontologx.metrics.ttp_metrics import TacticsMetrics

NEO4J_URL = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"  # noqa: S105
TESTS_FILE = "../resources/data/polito/test.ttl"

load_dotenv("../.env")

store = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

In [2]:
import neo4j
from langchain_core.documents import Document

from ontologx.store import GraphDocument, Node, Relationship
from ontologx.store.neo4j.utils import normalize_output_graph


def get_subgraph_from_node(node_uri: str) -> GraphDocument:
    """Get the subgraph of a node in the store.

    The subgraph will contain all the nodes and relationships connected to the given node, even indirectly.

    Args:
        node_uri (str): The URI of the node to get the subgraph from.

    Returns:
        GraphDocument: The subgraph of the node, with nodes and relationships.

    Raises:
        ValueError: If no subgraph is found for the given node URI.

    """
    # Ugly but quite efficient.
    nodes_subgraphs = store.query(
        """
        MATCH (n {uri: $node_uri})
        CALL apoc.path.subgraphAll(n,
            {labelFilter: '-mlsx__OutputDataset|-mlsx__ExampleDataset|-mlsx__TestDataset'})
        YIELD nodes, relationships
        RETURN
        [node IN nodes | {
        uri: node.uri,
        type: HEAD([label IN LABELS(node) WHERE label <> 'Resource']),
        properties: PROPERTIES(node)
        }] AS nodes,
        [rel IN relationships | {
        source: STARTNODE(rel).uri,
        target: ENDNODE(rel).uri,
        type: TYPE(rel)
        }] AS relationships
        """,
        params={"node_uri": node_uri},
    )

    if not nodes_subgraphs:
        msg = f"No subgraph found for node with URI: {node_uri}"
        raise ValueError(msg)

    nodes_subgraph = nodes_subgraphs[0]

    # Remove the DatasetRow node, as it is not needed in the output graph.
    # However it contains the event message, so it is used as the source of the graph.
    dataset_row_node = next(node for node in nodes_subgraph["nodes"] if node["type"] == "mlsx__DatasetRow")
    nodes_subgraph["nodes"].remove(dataset_row_node)
    nodes_subgraph["relationships"] = [
        relationship
        for relationship in nodes_subgraph["relationships"]
        if relationship["source"] != dataset_row_node["uri"] and relationship["target"] != dataset_row_node["uri"]
    ]

    # The neo4j date and time objects are quite problematic, as they are not JSON serializable.
    # This is a workaround to convert them to strings.
    for node in nodes_subgraph["nodes"]:
        for key, value in node["properties"].items():
            if isinstance(value, neo4j.time.DateTime | neo4j.time.Date | neo4j.time.Time):
                node["properties"][key] = value.iso_format()

    def get_node_id(uri: str) -> str:
        """Get the node id from the node uri."""
        final_str = uri.split("/")[-1]

        if "#" in final_str:
            return final_str.split("#")[-1]

        return final_str

    nodes_dict = {
        node["uri"]: Node(id=get_node_id(node["uri"]), type=node["type"], properties=node["properties"])
        for node in nodes_subgraph["nodes"]
    }

    relationships = (
        [
            Relationship(
                source=nodes_dict[relationship["source"]],
                target=nodes_dict[relationship["target"]],
                type=relationship["type"],
            )
            for relationship in nodes_subgraph["relationships"]
        ]
        if "relationships" in nodes_subgraph
        else []  # The node may not have any relationships
    )

    # Create the context from the event source, if present.
    source_node = next((node for node in nodes_dict.values() if node.type == "olx__Source"), None)
    context = {key: value for key, value in source_node.properties.items() if key != "uri"} if source_node else {}

    return GraphDocument(
        nodes=list(nodes_dict.values()),
        relationships=relationships,
        source=Document(
            page_content=dataset_row_node["properties"]["mlsx__eventMessage"],
            metadata={"context": context},
        ),
    )


def tests() -> list[GraphDocument]:
    """Return a list of test documents from the dataset."""
    test_nodes = store.query(
        """
        MATCH (d:mlsx__TestDataset)-[:mlsx__hasPart]->(r:mlsx__DatasetRow)-[:mlsx__hasLabel]->(e:olx__Event)
        RETURN r.mlsx__eventMessage as eventMessage, e.uri as uri, r.mlsx__hasTactic as tactics
        ORDER BY e.uri
        """,
    )
    tests = []
    for test in test_nodes:
        graph = get_subgraph_from_node(test["uri"])

        source_node = next((node for node in graph.nodes if node.type == "olx__Source"), None)
        context = {}
        if source_node:
            if source_node.properties.get("olx__sourceName"):
                context["sourceName"] = source_node.properties["olx__sourceName"]
            if source_node.properties.get("olx__sourceDevice"):
                context["sourceDevice"] = source_node.properties["olx__sourceDevice"]

        graph.source = Document(
            page_content=test["eventMessage"],
            metadata={"context": context, "tactics": test["tactics"]},
        )
        tests.append(normalize_output_graph(graph))

    return tests

In [3]:
y_true = tests()
y_pred: list[GraphDocument] = []

for graph in y_true:
    event_node = next((node for node in graph.nodes if node.type == "olx:Event"), None)
    if event_node is None:
        msg = "No Event node found in the graph"
        raise ValueError(msg)

    pred_event_node_uri = store.query(
        """
        MATCH (r:mlsx__Run)-[:mlsx__hasOutput]->(d:mlsx__OutputDataset)
        -[:mlsx__hasPart]->(dr:mlsx__DatasetRow)-[:mlsx__hasLabel]->(e:olx__Event)
        WHERE dr.mlsx__eventMessage = $event_message
        RETURN e.uri as uri
        """,
        params={
            "event_message": graph.source.page_content,
        },
    )

    pred_graph = get_subgraph_from_node(pred_event_node_uri[0]["uri"])
    y_pred.append(normalize_output_graph(pred_graph))

In [4]:
from pathlib import Path
from typing import cast

import pandas as pd

from ontologx.metrics.ttp_metrics import MITRETactic

llm = LLMFactory.create(backend="bedrock", model="us.anthropic.claude-sonnet-4-20250514-v1:0", temperature=0)

sessions_dict: dict[str, list[GraphDocument]] = {}
tactics_dict: dict[str, list[MITRETactic]] = {}
for pred, true in zip(y_pred, y_true, strict=True):
    event_node = next(e for e in pred.nodes if e.type == "olx:Event")
    session_id = cast("str | None", event_node.properties.get("olx:eventSessionID"))

    if session_id is None:
        continue

    if session_id not in sessions_dict:
        sessions_dict[session_id] = []
        tactics_dict[session_id] = []

    sessions_dict[session_id].append(pred)
    true_tactics = [MITRETactic(i.lower().title()) for i in true.source.metadata["tactics"]]
    tactics_dict[session_id] = list(set(tactics_dict[session_id] + true_tactics))

In [5]:
results = []

metrics = TacticsMetrics(
    y_pred_sessions=sessions_dict,
    y_true_tactics=tactics_dict,
    llm=llm,
    prompt_predict_tactics=Path("../resources/prompts/tactics.system.md").read_text()
)

print("Precision:", metrics.precision)
print("Recall:", metrics.recall)
print("F1 Score:", metrics.f1_score)

for tactic in MITRETactic:
    print(f"{tactic.name} Precision:", metrics.tactics_precision.get(tactic, 0.0))
    print(f"{tactic.name} Recall:", metrics.tactics_recall.get(tactic, 0.0))
    print(f"{tactic.name} F1 Score:", metrics.tactics_f1_score.get(tactic, 0.0))

Precision: 0.7322981366459628
Recall: 0.9730848861283643
F1 Score: 0.8335107956225969
EXECUTION Precision: 0.07051282051282051
EXECUTION Recall: 1.0
EXECUTION F1 Score: 0.1317365269461078
DISCOVERY Precision: 1.0
DISCOVERY Recall: 0.75
DISCOVERY F1 Score: 0.8571428571428571
INITIAL_ACCESS Precision: 1.0
INITIAL_ACCESS Recall: 1.0
INITIAL_ACCESS F1 Score: 1.0
PERSISTENCE Precision: 0.984
PERSISTENCE Recall: 1.0
PERSISTENCE F1 Score: 0.9919354838709677
EXFILTRATION Precision: 0.0
EXFILTRATION Recall: 0.0
EXFILTRATION F1 Score: 0.0
IMPACT Precision: 0.0
IMPACT Recall: 0.0
IMPACT F1 Score: 0.0
DEFENSE_EVASION Precision: 0.6666666666666666
DEFENSE_EVASION Recall: 0.5
DEFENSE_EVASION F1 Score: 0.5714285714285715
PRIVILEGE_ESCALATION Precision: 1.0
PRIVILEGE_ESCALATION Recall: 0.991869918699187
PRIVILEGE_ESCALATION F1 Score: 0.9959183673469388
CREDENTIAL_ACCESS Precision: 0.0
CREDENTIAL_ACCESS Recall: 0.0
CREDENTIAL_ACCESS F1 Score: 0.0
LATERAL_MOVEMENT Precision: 0.0
LATERAL_MOVEMENT Recall: